In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from ld_estimates.estimators import r2
from ld_estimates.calibration import build_calibration_models
from ld_estimates import create_calibrated_estimator
from paper.experiments import simulate_bias_curve_data
from paper.plotting import plot_bias_curves

In [ ]:
sample_sizes = [5, 10, 25]
r2_grid = np.linspace(0, 1, 21)
p_s, p_t = 0.5, 0.5

curves = {"Diploid": {}, "Pseudohaploid": {}}
for n in sample_sizes:
    curves["Diploid"][n]       = simulate_bias_curve_data(n, p_s, p_t, r2_grid, Nrep=500, r2_estimator=r2)
    curves["Pseudohaploid"][n] = simulate_bias_curve_data(n, p_s, p_t, r2_grid, Nrep=500, r2_estimator=r2, pseudohaploid=True)

In [ ]:
fig, axes = plot_bias_curves(curves, sample_sizes, show_spread=True)
plt.show()

In [ ]:
sample_sizes = [5, 10, 25]
r2_grid = np.linspace(0, 1, 21)

# Build calibration models for each n — slow cell, run once
# Nrep=1000 is fast enough to see the shape; use 5000 for the final figure
master_diploid = {"r2": {}}
master_pseudo  = {"r2": {}}

for n in sample_sizes:
    master_diploid["r2"][n] = build_calibration_models(
        n, N_replicates=1000, r2_grid_to_model=r2_grid,
        estimators_to_calibrate={"r2": r2}, pseudohaploid=False,
    )["r2"]
    master_pseudo["r2"][n] = build_calibration_models(
        n, N_replicates=1000, r2_grid_to_model=r2_grid,
        estimators_to_calibrate={"r2": r2}, pseudohaploid=True,
    )["r2"]

In [ ]:
# Create calibrated estimators for both data types
r2_cal_d   = create_calibrated_estimator(r2, "r2", master_diploid, calibration_type="cal")
r2_mcal_d  = create_calibrated_estimator(r2, "r2", master_diploid, calibration_type="indep")
r2_cal_p   = create_calibrated_estimator(r2, "r2", master_pseudo,  calibration_type="cal",   pseudohaploid=True)
r2_mcal_p  = create_calibrated_estimator(r2, "r2", master_pseudo,  calibration_type="indep", pseudohaploid=True)

p_s, p_t = 0.5, 0.5

curves_cal = {
    "Diploid (raw)":        {},
    "Diploid (Cal)":        {},
    "Diploid (mCal)":       {},
    "Pseudohaploid (raw)":  {},
    "Pseudohaploid (Cal)":  {},
    "Pseudohaploid (mCal)": {},
}

for n in sample_sizes:
    curves_cal["Diploid (raw)"][n]        = simulate_bias_curve_data(n, p_s, p_t, r2_grid, Nrep=500, r2_estimator=r2)
    curves_cal["Diploid (Cal)"][n]        = simulate_bias_curve_data(n, p_s, p_t, r2_grid, Nrep=500, r2_estimator=r2_cal_d)
    curves_cal["Diploid (mCal)"][n]       = simulate_bias_curve_data(n, p_s, p_t, r2_grid, Nrep=500, r2_estimator=r2_mcal_d)
    curves_cal["Pseudohaploid (raw)"][n]  = simulate_bias_curve_data(n, p_s, p_t, r2_grid, Nrep=500, r2_estimator=r2,       pseudohaploid=True)
    curves_cal["Pseudohaploid (Cal)"][n]  = simulate_bias_curve_data(n, p_s, p_t, r2_grid, Nrep=500, r2_estimator=r2_cal_p, pseudohaploid=True)
    curves_cal["Pseudohaploid (mCal)"][n] = simulate_bias_curve_data(n, p_s, p_t, r2_grid, Nrep=500, r2_estimator=r2_mcal_p, pseudohaploid=True)

In [ ]:
# ylim=(-0.1, 1) to show mCal curves that go slightly negative
fig, axes = plot_bias_curves(curves_cal, sample_sizes, show_spread=False, ylim=(-0.1, 1))
plt.show()